In [29]:
from dotenv import load_dotenv
load_dotenv()
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import List,Literal
from langchain_groq import ChatGroq
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class MailState(BaseModel):
    query:str = Field(description="User query goes here")
    draft_mail:str = Field(default="")
    humanfeedback:str= Field(default="")
    send_mail:str= Field(default="")

llm = ChatGroq(model = "openai/gpt-oss-20b")
memory = InMemorySaver()
def draft_mail(state:MailState) -> MailState:
    if state.humanfeedback:
        draft = llm.invoke( f"""
        You are revising an existing email draft.

        Original user request:
        {state.query}

        Previous draft:
        {state.draft_mail}

        User's requested changes:
        {state.humanfeedback}

        Instructions:
        - Revise the previous draft according to the user's feedback.
        - Preserve all previously requested changes unless the user
          explicitly asks to change them.
        - Do not reintroduce content that the user asked you to remove.
        - Do not add placeholders unless explicitly requested.
        - Return only the revised email.
        """).content
    else:

        draft = llm.invoke(state.query).content

    state.draft_mail = draft
    return state

def human_feedback(state:MailState) -> MailState:
    #Get the state.draft mail and ask the user that the draft mail is fine or not if not go back to draft mail again
    feedback = interrupt({
        "draft_mail":state.draft_mail,
        "question":"Do you want to continue or re-draft the mail again?"
    })

    fb = (feedback or "").strip().lower()
    if fb in ["approved", "yes", "goahead"]:
        state.humanfeedback = ""
        return state
    else:
        state.humanfeedback = feedback
        return state

def send_mail(state:MailState) -> MailState:
    if state.humanfeedback:
        state.send_mail = state.humanfeedback
        return state
    else:
        print("Email sent successfully")
        state.send_mail = "Email Sent Successfully"
        return state

def conditional_route(state:MailState) -> Literal["draft_mail","send_mail"]:
    if state.humanfeedback:
        return "draft_mail"
    return "send_mail"

graph = StateGraph(MailState)
graph.add_node("draft_mail", draft_mail)
graph.add_node( "human_feedback", human_feedback)
graph.add_node("send_mail", send_mail)

graph.add_edge(START, "draft_mail")
graph.add_edge("draft_mail", "human_feedback")
graph.add_conditional_edges("human_feedback",conditional_route)
graph.add_edge("send_mail", END)

graph = graph.compile(checkpointer=memory)
config = {"configurable":{"thread_id":1}}


res = graph.invoke({"query":"Draft a sick leave mail for 27th Octobar for my Manager"},
                   config = config)
print(res["draft_mail"])




**Subject:** Sick Leave Request – 27 October 2024

Dear [Manager’s Name],

I hope you’re doing well. I am writing to inform you that I am feeling unwell today and will not be able to come to the office. I will be taking sick leave for **27 October 2024**.

I will keep you updated on my recovery and will be available via email or phone for any urgent matters that require my attention. I will make sure to catch up on any missed work as soon as I am able.

Thank you for your understanding.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]


In [ ]:
#response = graph.invoke(Command(resume="I don't want this draft"), config = config)
#print(response)

{'query': 'Draft a sick leave mail for 27th Octobar for my Manager', 'draft_mail': '**Subject:** Sick Leave Request – 27\u202fOct\u202f2023  \n\nDear [Manager’s Name],\n\nI hope you’re well. I’m writing to let you know that I’m feeling unwell today and will not be able to come into the office. I plan to take sick leave for **27\u202fOctober\u202f2023**.\n\nI will monitor my emails intermittently and will keep you updated on my recovery. If anything urgent comes up, please feel free to contact me via phone or text at [your phone number].\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Contact Information]', 'humanfeedback': "I don't want this draft", 'send_mail': "I don't want this draft"}


In [30]:
response = graph.invoke(Command(resume="yes"), config = config)
print(response["draft_mail"])
print(response["humanfeedback"])


Email sent successfully
**Subject:** Sick Leave Request – 27 October 2024

Dear [Manager’s Name],

I hope you’re doing well. I am writing to inform you that I am feeling unwell today and will not be able to come to the office. I will be taking sick leave for **27 October 2024**.

I will keep you updated on my recovery and will be available via email or phone for any urgent matters that require my attention. I will make sure to catch up on any missed work as soon as I am able.

Thank you for your understanding.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]

